# Day 4 — Winter ONI forecast + hybrid impacts grid

1. Forecast winter ONI with the trained CNN  
2. Precompute ONI→winter impacts on a **lat/lon grid** (hybrid: click-anywhere later via interpolation)  
3. Also compute a short **named city** list for map labels  
4. Write `impacts.json`

**Winter rule:** Dec belongs to the *next* winter year (Dec 2026 → winter 2027).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarib-sami/ninonet/blob/main/enso/day4_impacts.ipynb)

Run **Day 3 tuned** first if you can (`enso_cnn_lead6_tuned.pt`). Falls back to untuned checkpoints.

**Grid size:** default `STEP=4` over North America (~200–300 points, ~15–40 min). Use `STEP=5` if short on time; `STEP=3` for finer.


## 0. Install


In [ ]:
!pip install -q xarray netCDF4 numpy pandas scikit-learn matplotlib requests torch pyarrow


## 1. Mount Drive


In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/ensocast/data")
OUT_DIR = Path("/content/drive/MyDrive/ensocast/artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / "pacific_anom.nc").exists(), "Rerun Day 1"
assert (DATA_DIR / "oni_monthly.csv").exists(), "Rerun Day 1"
print("Data:", DATA_DIR)
print("Artifacts:", OUT_DIR)


## 2. Load SST anomalies + ONI


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

anom = xr.open_dataarray(DATA_DIR / "pacific_anom.nc")
if isinstance(anom, xr.Dataset):
    anom = anom[list(anom.data_vars)[0]]

oni_df = pd.read_csv(DATA_DIR / "oni_monthly.csv", parse_dates=["time"])

arr = anom.values.astype("float32")
times = pd.to_datetime(anom["time"].values).to_period("M").to_timestamp()

oni_series = oni_df.set_index("time")["oni"]
oni_series.index = pd.to_datetime(oni_series.index).to_period("M").to_timestamp()
oni = oni_series.reindex(times).to_numpy(dtype="float32")

missing = int(np.isnan(oni).sum())
if missing:
    print(f"Dropping {missing} month(s) with no ONI.")
    valid = ~np.isnan(oni)
    arr, times, oni = arr[valid], times[valid], oni[valid]

print("months:", len(arr), "last:", times[-1].date(), "last ONI:", float(oni[-1]))


## 3. Load CNN and forecast winter ONI

Default **lead 6** (beat persistence in Day 3). Set `LEAD = 3` to prefer that checkpoint.


In [ ]:
import torch
import torch.nn as nn

LEAD = 6
WINDOW = 12
WINTER_LABEL = "2026-27"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ENSOForecaster(nn.Module):
    def __init__(self, in_months=12, dropout=0.4, use_bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_months, 32, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(32))
        layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(64))
        layers += [
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


candidates = [
    OUT_DIR / f"enso_cnn_lead{LEAD}_tuned.pt",
    OUT_DIR / f"enso_cnn_lead{LEAD}.pt",
    OUT_DIR / "enso_cnn_lead3_tuned.pt",
    OUT_DIR / "enso_cnn_lead3.pt",
]
ckpt_path = next((p for p in candidates if p.exists()), None)
assert ckpt_path is not None, "No CNN checkpoint — rerun Day 2/3"
print("Loading", ckpt_path)

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt["model_state"]
use_bn = any("running_mean" in k for k in state)
dropout = float(ckpt.get("hp", {}).get("dropout", 0.4)) if isinstance(ckpt.get("hp"), dict) else 0.4

model = ENSOForecaster(dropout=dropout, use_bn=use_bn).to(device)
model.load_state_dict(state)
model.eval()
print("use_bn:", use_bn, "ckpt lead:", ckpt.get("lead"), "test_rmse:", ckpt.get("test_rmse"))


In [ ]:
x = arr[-WINDOW:].astype("float32")
mu, sd = ckpt.get("norm_mu"), ckpt.get("norm_sd")
if mu is not None and sd is not None:
    x = (x - float(mu)) / float(sd)
    print(f"applied norm mu={mu:.4f} sd={sd:.4f}")

x_t = torch.from_numpy(x[None]).to(device)
with torch.no_grad():
    forecast_oni = float(model(x_t).cpu().numpy().squeeze())

last_month = times[-1]
print(f"input ends: {last_month.date()}")
print(f"forecast ONI (winter {WINTER_LABEL}): {forecast_oni:.3f}")


## 4. Build the hybrid grid + label cities

- **Grid points:** every `STEP` degrees in the box — precomputed impacts for click interpolation on Day 5  
- **Cities:** named pins only (looked up from nearest grid point later, or computed directly)


In [ ]:
# North America box (widen if you want). STEP=5 faster; STEP=3 finer.
LAT_MIN, LAT_MAX = 15.0, 65.0
LON_MIN, LON_MAX = -140.0, -50.0
STEP = 4.0

lats = np.arange(LAT_MIN, LAT_MAX + 1e-6, STEP)
lons = np.arange(LON_MIN, LON_MAX + 1e-6, STEP)
grid_points = [(float(la), float(lo)) for la in lats for lo in lons]
print(f"grid: {len(lats)} lats x {len(lons)} lons = {len(grid_points)} points")

# Named cities for labels on the map (still useful UX)
CITIES = [
    ("Vancouver", 49.28, -123.12),
    ("Seattle", 47.61, -122.33),
    ("San Francisco", 37.77, -122.42),
    ("Los Angeles", 34.05, -118.24),
    ("Phoenix", 33.45, -112.07),
    ("Denver", 39.74, -104.99),
    ("Calgary", 51.05, -114.07),
    ("Winnipeg", 49.90, -97.14),
    ("Minneapolis", 44.98, -93.27),
    ("Chicago", 41.88, -87.63),
    ("Toronto", 43.65, -79.38),
    ("Montreal", 45.50, -73.57),
    ("Boston", 42.36, -71.06),
    ("New York", 40.71, -74.01),
    ("Washington DC", 38.91, -77.04),
    ("Atlanta", 33.75, -84.39),
    ("Miami", 25.76, -80.19),
    ("New Orleans", 29.95, -90.07),
    ("Houston", 29.76, -95.37),
    ("Dallas", 32.78, -96.80),
    ("Mexico City", 19.43, -99.13),
    ("Anchorage", 61.22, -149.90),
]

# Merge: compute impacts for grid + any city not near a grid node
locations = []  # (id, name_or_None, lat, lon)
for la, lo in grid_points:
    locations.append((f"g:{la:.2f},{lo:.2f}", None, la, lo))
for name, la, lo in CITIES:
    locations.append((f"c:{name}", name, la, lo))

print("total fetch targets:", len(locations), "(grid + cities)")


## 5. Open-Meteo winters + DJF ONI

Winter year **Y** = Dec (Y−1) + Jan Y + Feb Y.


In [ ]:
import time
import requests

START, END = "1960-01-01", "2025-12-31"
ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"


def fetch_daily(lat, lon, retries=4):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START,
        "end_date": END,
        "daily": "temperature_2m_mean,precipitation_sum,snowfall_sum",
        "timezone": "auto",
    }
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(ARCHIVE, params=params, timeout=120)
            if r.status_code == 200:
                return r.json()
            last_err = RuntimeError(f"HTTP {r.status_code}")
        except Exception as e:
            last_err = e
        time.sleep(1.5 * (attempt + 1))
    raise last_err


def daily_to_winters(payload):
    daily = payload["daily"]
    df = pd.DataFrame(
        {
            "date": pd.to_datetime(daily["time"]),
            "tmean": daily["temperature_2m_mean"],
            "precip": daily["precipitation_sum"],
            "snow": daily["snowfall_sum"],
        }
    )
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["winter_year"] = np.where(df["month"] == 12, df["year"] + 1, df["year"])
    winter = df[df["month"].isin([12, 1, 2])]
    g = winter.groupby("winter_year").agg(
        tmean=("tmean", "mean"),
        precip=("precip", "sum"),
        snow=("snow", "sum"),
        n_days=("tmean", "count"),
    )
    return g[g["n_days"] >= 85]


oni_by_winter = (
    oni_df.assign(time=pd.to_datetime(oni_df["time"]))
    .assign(month=lambda d: d["time"].dt.month, year=lambda d: d["time"].dt.year)
    .loc[lambda d: d["month"] == 1]
    .set_index("year")["oni"]
    .astype(float)
)
print("ONI winters:", int(oni_by_winter.index.min()), "->", int(oni_by_winter.index.max()))


In [ ]:
# Resume-friendly cache on Drive
cache_path = OUT_DIR / "winter_cache.parquet"
if cache_path.exists():
    cache_df = pd.read_parquet(cache_path)
    print("loaded cache", cache_path, "rows", len(cache_df))
else:
    cache_df = pd.DataFrame(columns=["loc_id", "winter_year", "tmean", "precip", "snow"])

cached_ids = set(cache_df["loc_id"].unique()) if len(cache_df) else set()
print("already cached locations:", len(cached_ids))

new_rows = []
failures = []
for i, (loc_id, name, lat, lon) in enumerate(locations, 1):
    if loc_id in cached_ids:
        continue
    print(f"[{i}/{len(locations)}] fetch {loc_id} ...", end=" ")
    try:
        winters = daily_to_winters(fetch_daily(lat, lon))
        part = winters.reset_index().rename(columns={"winter_year": "winter_year"})
        part.insert(0, "loc_id", loc_id)
        new_rows.append(part[["loc_id", "winter_year", "tmean", "precip", "snow"]])
        print(len(winters), "winters")
    except Exception as e:
        print("FAIL", e)
        failures.append((loc_id, str(e)))
    time.sleep(0.35)
    # periodic save
    if new_rows and (len(new_rows) % 25 == 0):
        cache_df = pd.concat([cache_df, *new_rows], ignore_index=True)
        cache_df.to_parquet(cache_path, index=False)
        cached_ids = set(cache_df["loc_id"].unique())
        new_rows = []
        print("  checkpoint cache ->", cache_path)

if new_rows:
    cache_df = pd.concat([cache_df, *new_rows], ignore_index=True)
cache_df.to_parquet(cache_path, index=False)
print("cache rows:", len(cache_df), "unique locs:", cache_df["loc_id"].nunique())
print("failures:", len(failures))


## 6. Fit ONI → anomalies at every location

Same idea as before: past DJF ONI vs that point’s winter anomalies; plug in `forecast_oni`.


In [ ]:
def phrase(var, anom):
    if var == "temp":
        if anom > 0.3:
            return "warmer than usual"
        if anom < -0.3:
            return "colder than usual"
        return "near-normal temperatures"
    if var == "precip":
        if anom > 10:
            return "wetter than usual"
        if anom < -10:
            return "drier than usual"
        return "near-normal precipitation"
    if anom > 5:
        return "snowier than usual"
    if anom < -5:
        return "less snow than usual"
    return "near-normal snowfall"


def confidence_tag(score):
    if score >= 0.7:
        return "strong"
    if score >= 0.55:
        return "moderate"
    return "mixed"


def fit_location(winters, forecast_oni, oni_by_winter):
    common = winters.join(oni_by_winter.rename("oni"), how="inner").dropna()
    if len(common) < 20:
        return None
    confidences = []
    out = {"n_winters": int(len(common))}
    for var, key in [("temp", "tmean"), ("precip", "precip"), ("snow", "snow")]:
        clim = float(common[key].mean())
        anom = common[key] - clim
        b, a = np.polyfit(common["oni"].to_numpy(), anom.to_numpy(), 1)
        pred = float(a + b * forecast_oni)
        enso = common[common["oni"] >= 0.5]
        if len(enso) >= 5 and abs(pred) > 1e-6:
            conf = float((np.sign(enso[key] - clim) == np.sign(pred)).mean())
        else:
            conf = 0.5
        confidences.append(conf)
        out[f"{var}_anom"] = pred
        out[f"{var}_phrase"] = phrase(var, pred)
        out[f"{var}_slope"] = float(b)
    out["confidence"] = float(np.mean(confidences))
    out["confidence_tag"] = confidence_tag(out["confidence"])
    return out


meta = {loc_id: (name, lat, lon) for loc_id, name, lat, lon in locations}
grid_impacts = []
city_impacts = []

for loc_id, group in cache_df.groupby("loc_id"):
    winters = group.set_index("winter_year")[["tmean", "precip", "snow"]]
    fitted = fit_location(winters, forecast_oni, oni_by_winter)
    if fitted is None or loc_id not in meta:
        continue
    name, lat, lon = meta[loc_id]
    row = {
        "lat": lat,
        "lon": lon,
        "forecast_oni": float(forecast_oni),
        "winter": WINTER_LABEL,
        **fitted,
    }
    if name is None:
        grid_impacts.append(row)
    else:
        row = {"city": name, **row}
        city_impacts.append(row)

print("grid points with fits:", len(grid_impacts))
print("cities with fits:", len(city_impacts))


## 7. Interpolation helper (what Day 5 will use)

Inverse-distance weighting over the precomputed grid — click any lat/lon without a new API call.


In [ ]:
def interpolate_impact(lat, lon, grid, k=4, power=2.0):
    if not grid:
        raise ValueError("empty grid")
    lats = np.array([g["lat"] for g in grid], dtype=float)
    lons = np.array([g["lon"] for g in grid], dtype=float)
    # crude degrees distance (OK for regional map)
    d = np.hypot(lats - lat, lons - lon)
    d = np.maximum(d, 1e-6)
    idx = np.argsort(d)[:k]
    w = 1.0 / (d[idx] ** power)
    w /= w.sum()

    def gather(key):
        return float(np.sum(w * np.array([grid[i][key] for i in idx], dtype=float)))

    temp = gather("temp_anom")
    precip = gather("precip_anom")
    snow = gather("snow_anom")
    conf = gather("confidence")
    return {
        "lat": lat,
        "lon": lon,
        "temp_anom": temp,
        "precip_anom": precip,
        "snow_anom": snow,
        "temp_phrase": phrase("temp", temp),
        "precip_phrase": phrase("precip", precip),
        "snow_phrase": phrase("snow", snow),
        "confidence": conf,
        "confidence_tag": confidence_tag(conf),
        "winter": WINTER_LABEL,
        "forecast_oni": float(forecast_oni),
        "source": "idw_grid",
    }


# smoke test: Toronto should be close to its city fit if both exist
demo = interpolate_impact(43.65, -79.38, grid_impacts)
print("IDW @ Toronto:", {k: demo[k] for k in ["temp_anom", "precip_anom", "snow_anom", "confidence_tag"]})
if city_impacts:
    t = next((c for c in city_impacts if c["city"] == "Toronto"), None)
    if t:
        print("direct Toronto:", {k: t[k] for k in ["temp_anom", "precip_anom", "snow_anom", "confidence_tag"]})


## 8. Write `impacts.json`


In [ ]:
import json

payload = {
    "winter": WINTER_LABEL,
    "forecast_oni": float(forecast_oni),
    "model_checkpoint": ckpt_path.name,
    "model_lead": int(ckpt.get("lead", LEAD)),
    "input_end": str(last_month.date()),
    "grid_meta": {
        "lat_min": LAT_MIN,
        "lat_max": LAT_MAX,
        "lon_min": LON_MIN,
        "lon_max": LON_MAX,
        "step": STEP,
        "n_points": len(grid_impacts),
        "interpolation": "inverse_distance_weighting_k4",
    },
    "disclaimer": (
        "Model forecast fed through a historical ONI–local winter relationship on a precomputed grid. "
        "Click-anywhere values are interpolated. Not an official outlook; mid-latitude signal can be weak."
    ),
    "grid": grid_impacts,
    "cities": city_impacts,
}

for path in [OUT_DIR / "impacts.json", DATA_DIR / "impacts.json"]:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("Wrote", path)

print(
    f"winter {WINTER_LABEL}  ONI={forecast_oni:.3f}  "
    f"grid={len(grid_impacts)}  cities={len(city_impacts)}"
)
print("Day 4 checkpoint: hybrid impacts grid ready.")


## 9. Preview map (grid colored by temp anomaly)


In [ ]:
import matplotlib.pyplot as plt

gdf = pd.DataFrame(grid_impacts)
fig, ax = plt.subplots(figsize=(11, 5))
sc = ax.scatter(
    gdf["lon"], gdf["lat"], c=gdf["temp_anom"],
    cmap="RdBu_r", vmin=-2, vmax=2, s=40, edgecolors="none",
)
if city_impacts:
    cdf = pd.DataFrame(city_impacts)
    ax.scatter(cdf["lon"], cdf["lat"], c="k", s=18, marker="x", label="cities")
    for _, r in cdf.iterrows():
        ax.annotate(r["city"], (r["lon"], r["lat"]), fontsize=7, alpha=0.8)
ax.legend(loc="lower left")
plt.colorbar(sc, ax=ax, label="temp anomaly (C)")
ax.set_title(f"Winter {WINTER_LABEL} grid | ONI={forecast_oni:.2f} | step={STEP}")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
ax.set_xlim(LON_MIN - 2, LON_MAX + 2)
ax.set_ylim(LAT_MIN - 2, LAT_MAX + 2)
plt.tight_layout()
fig.savefig(OUT_DIR / "impacts_grid_preview.png", dpi=140)
plt.show()
print("Wrote", OUT_DIR / "impacts_grid_preview.png")
